# Play with fine tuned models

In [25]:
import os
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig
# from datetime import datetime
# import matplotlib.pyplot as plt
# from dotenv import load_dotenv
import numpy as np
from torch.nn import functional as F

## Load base model and adaptors

In [ ]:
hf_token = ""  # removed in public repo
login(hf_token, add_to_git_credential=True)

"baby-gpt-sft-tinystories-v1": r = 16, attn only, wandb 2026-04-24_22.15.40

"baby-gpt-sft-tinystories-v2": r = 32, attn only, wandb 2026-04-25_04.15.36

"baby-gpt-sft-tinystories-v3": r = 32, attn + mlp, wandb 2026-04-25_07.28.57

"baby-gpt-sft-tinystories-v4": r = 64, attn + mlp, wandb 2026-04-25_10.02.20

In [ ]:
models = [
    "littleBallOfFur/baby-gpt-base",
    "littleBallOfFur/baby-gpt-sft-tinystories-v1",
    "littleBallOfFur/baby-gpt-sft-tinystories-v2",
    "littleBallOfFur/baby-gpt-sft-tinystories-v3",
    "littleBallOfFur/baby-gpt-sft-tinystories-v4",
]

In [76]:
tokenizer = AutoTokenizer.from_pretrained(models[0],trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [21]:
model_0 = AutoModelForCausalLM.from_pretrained(
    models[0], 
    trust_remote_code=True
)
model_1 = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(models[0],
    trust_remote_code=True), models[1]
)
model_2 = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(models[0],
    trust_remote_code=True), models[2]
)
model_3 = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(models[0],
    trust_remote_code=True), models[3]
)
model_4 = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(models[0],
    trust_remote_code=True), models[4]
)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

...

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

In [71]:
model_4

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): BabyGPTForCausalLM(
      (transformer): ModuleDict(
        (wte): Embedding(50304, 768)
        (wpe): Embedding(1024, 768)
        (h): ModuleList(
          (0-11): 12 x Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): CausalSelfAttention(
              (c_attn): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=2304, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lo

## Test generation

In [ ]:
def test_generate(model, prompts = [" "], max_new = 32, auto=True,temperature=1.0 ):
    torch.manual_seed(42)
    model.eval()
    # if generate in batch the padded ones are bad
    for i in range(len(prompts)):
        input_tokens = tokenizer.encode(prompts[i], return_tensors='pt')
        if auto:
            with torch.no_grad():
                output_tokens = model.generate(
                    input_tokens,
                    max_new_tokens=max_new,
                    do_sample=True,
                    temperature=temperature,
                    top_k=50,
                    top_p=0.95,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
        else:
            for _ in range(max_new):
                # forward the model to get the logits
                with torch.no_grad():
                    logits = model(input_tokens).logits 
                    logits = logits[:, -1, :]/temperature # (B=1, vocab_size)
                    probs = F.softmax(logits, dim=-1)
                    topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
                    ix = torch.multinomial(topk_probs, 1) # (B, 1)
                    new_tok = torch.gather(topk_indices, -1, ix) # (B, 1)
                    input_tokens = torch.cat((input_tokens, new_tok), dim=1)

                    # v, _ = torch.topk(logits, min(50, logits.size(-1)))
                    # logits[logits < v[:, [-1]]] = -float('Inf')
                    # probs = F.softmax(logits, dim=-1)
                    # new_tok = torch.multinomial(probs, num_samples=1)
                    # input_tokens = torch.cat((input_tokens, new_tok), dim=1)
            output_tokens = input_tokens
        
        print(f'\n{prompts[i]}:\n{tokenizer.decode(output_tokens[0])}')



In [84]:
# torch.manual_seed(42)
prompts = [
    # "Once upon a time", 
    # "There was a little fat", 
    # "In a forest far far away",
    "A big fat squirrel"
]

test_generate(model_1,prompts,auto=True, max_new = 256 )

test_generate(model_2,prompts,auto=True, max_new = 256 )

test_generate(model_3,prompts,auto=True, max_new = 256 )

test_generate(model_4,prompts,auto=True, max_new = 256 )



A big fat squirrel:
A big fat squirrel was playing in the park. Suddenly, a big big dog came out of the bushes and started jumping. The big fat squirrel was so excited he jumped out of the bushes. He started to laugh and laughed until he heard the dog bark, "Hank!" Then, the dog barked "Papa!" "Hi!" and the big fat squirrel started to get scared. He wanted to be friends with the big dog but it was too late.
"Hank!" he cried. "Don't worry, we'll get home soon." The big fat squirrel went back to his home and he knew he'd been sad for so long. He thought back to his favorite toy. He ran out of his backyard and hugged his toys. He thought goodbye to his puppy and knew he was home. He had a lot of fun with his big fat squirrel and the big dog in the park.<|endoftext|>

A big fat squirrel:
A big fat squirrel was playing in the park. He had a red ball that he had never seen before. He was so excited to play with the ball that he threw it high up in the sky. Suddenly, a bird flew down and lan

## Test story relay

In [100]:
def story_relay(models, prompt, each_turn_tokens=32, max_round=-1):
    for model in models:
        model.eval()
    tokens_so_far = tokenizer.encode(prompt, return_tensors='pt')
    print(f"\n{prompt}")
    if max_round > 0:
        for turn in range(max_round*len(models)):
            current_model = models[turn % len(models)] 
            with torch.no_grad():
                pre_length = tokens_so_far.shape[1]
                tokens_so_far = current_model.generate(
                    input_ids = tokens_so_far, 
                    max_new_tokens = each_turn_tokens,
                    do_sample = True,
                    top_k = 50,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_tokens = tokens_so_far[0,pre_length:]
            print(f"\nModel {turn % len(models) + 1} says:{tokenizer.decode(new_tokens)}")
    else:
        turn = 0
        while tokens_so_far.shape[1]<=1024-each_turn_tokens:
            current_model = models[turn % len(models)] 
            with torch.no_grad():
                pre_length = tokens_so_far.shape[1]
                tokens_so_far = current_model.generate(
                    input_ids = tokens_so_far, 
                    max_new_tokens = each_turn_tokens,
                    do_sample = True,
                    top_k = 50,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_tokens = tokens_so_far[0,pre_length:]
            print(f"\nModel {turn % len(models) + 1} says:{tokenizer.decode(new_tokens)}")
            turn += 1
            if tokenizer.eos_token_id in new_tokens.detach().cpu().tolist():
                break

    print("="*40)
    print(f"The full story is:\n{tokenizer.decode(tokens_so_far[0])}")
    print(f"Total number of tokens: {tokens_so_far.shape[1]}")


In [105]:
prompt = "A big fat squirrel jumps"
story_relay(
    [model_1, model_2, model_3, model_4], 
    prompt, 
    each_turn_tokens=16, max_round=-1
)


A big fat squirrel jumps

Model 1 says: above, looking for food. Suddenly, a voice says: Squirrel! Squirrel can

Model 2 says: only eat nuts if you break him. One day, he disappears into the forest

Model 3 says:. He drops down and finds a box.
He asks the box: "

Model 4 says:What is inside?". He tells it: "It has a lot of nuts and

Model 1 says: seeds. We need them to survive".
The box says: "Nuts

Model 2 says:!" A squirrel is very surprised and says: "I don't know! I

Model 3 says:'m so hungry". He picks up some nuts and begins to eat. After a

Model 4 says: few minutes, he feels the crunch of the nuts.
The squirrel eats all

Model 1 says: the nuts and feels full. He tells his friends: "It's okay.

Model 2 says: Let me put a plate on top." The little squirrel says: "Yes!

Model 3 says: I will eat some more."
The little squirrel hugs the big fat squirrel and

Model 4 says: says: "Thank you!" The box says: "I am sorry. I

Model 1 says: didn't say what I wanted." The squirrel wins an